# Add Special Training Clips [optional]

This notebook is useful for adding small segments of videos as special training data to boost the model performance in weaker areas.

By specifying a separate backup directory, these files can easily be copied into different data versions when required.

Although only small sections of the audio are needed, it is more efficient to download the whole audio file, segment it and discard the segments that do not fall within the specified time range.

## Imports

In [ ]:
import re
import shutil
from pathlib import Path

## SETTINGS (adjust as required)

Specify version for data and model

In [ ]:
VERSION = "v2"  # Specify version number here

Data Directories

In [ ]:
DATA_DIR = Path("..") / "data" / VERSION

RAW_DATA_DIR = DATA_DIR / "raw" 
CLIPS_DATA_DIR = DATA_DIR / "clips"
RESULTS_DIR = DATA_DIR / "results"

STAGING_DIR = CLIPS_DATA_DIR / "segments"
MUSIC_CLIPS_DIR = CLIPS_DATA_DIR / "music"
NOT_MUSIC_CLIPS_DIR = CLIPS_DATA_DIR / "not-music"

# === Create the folders if they don't exist ===
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
STAGING_DIR.mkdir(parents=True, exist_ok=True)
MUSIC_CLIPS_DIR.mkdir(parents=True, exist_ok=True)
NOT_MUSIC_CLIPS_DIR.mkdir(parents=True, exist_ok=True)

Backup directory (useful for copying into other data versions later)

In [ ]:
BACKUP_DIR = Path("..") / "data" / "backup" / "acapella_clips"  # or None

# === Create the folders if they don't exist ===
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

Output Audio filepath

In [ ]:
OUTPUT_AUDIO = RAW_DATA_DIR / "training_audio.m4a"
OUTPUT_AUDIO_CLIP = RAW_DATA_DIR / "training_audio_clip.m4a"

Youtube video settings

In [ ]:
URL = "https://www.youtube.com/watch?v=-pcP7RjS6JY"

clip_start = 3575  # seconds
clip_end = 3585  # seconds
is_music = True

Clip size

In [ ]:
CLIP_SIZE = 5  # seconds

## Download Audio for YouTube Video

Download audio (whole video)

In [ ]:
!yt-dlp -q --force-overwrites -f "worstaudio[ext=m4a]/worstaudio" -o "{OUTPUT_AUDIO}" {URL}

## Split Audio File into 5 Second Clips

Get video_id to use in clips filename (allows multiple videos to be used as training data)

In [ ]:
def extract_safe_id(url: str) -> str:
    match = re.search(r"v=([^&]+)", url)
    if not match:
        return ""
    raw = match.group(1)
    return re.sub(r"[^A-Za-z0-9-]", "", raw)

video_id = extract_safe_id(URL)
video_id

Split into 5 second clips

In [ ]:
!ffmpeg -loglevel error -i "{OUTPUT_AUDIO_CLIP}" -f segment -segment_time {CLIP_SIZE} -c copy "{STAGING_DIR}/clip-{video_id}_%03d.m4a"

## Move Clips Within Specified Time Window

In [ ]:
# 1. List files inside STAGING_DIR with names starting with 'clip_'
files = list(STAGING_DIR.glob('clip*.m4a'))

moved_count = 0

for file_path in files:
    try:
        # 2. Extract number using pathlib's .stem (filename without extension)
        # "out_005.m4a" -> "out_005" -> split -> "005"
        file_num = int(file_path.stem.split('_')[1])
        
        # 3. Calculate timestamp
        clip_time = file_num * CLIP_SIZE

        # 4. Only move clips within desired time range
        if clip_start <= clip_time < clip_end:    
            # 5. Determine destination
            target_dir = MUSIC_CLIPS_DIR if is_music else NOT_MUSIC_CLIPS_DIR
            dest_path = target_dir / file_path.name
            
            # 6. Copy and move the file
            if BACKUP_DIR:
                shutil.copy2(file_path, BACKUP_DIR)
            shutil.move(file_path, dest_path)
            moved_count += 1
            
    except (ValueError, IndexError) as e:
        print(f"Skipping {file_path.name}: Could not parse number. Error: {e}")

print(f"Done! Moved {moved_count} files into {CLIPS_DATA_DIR} sub-directories.")


Remove the staging directory (which includes clips that aren't needed)

In [ ]:
shutil.rmtree(STAGING_DIR, ignore_errors=True)
print(f"Cleaned up: {STAGING_DIR} has been removed.")